In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.redis import RedisSaver
from langchain_ollama import ChatOllama
from typing import TypedDict, Annotated
from operator import add
from langchain_core.messages  import SystemMessage, HumanMessage, AIMessage

In [2]:
llm =ChatOllama(model="llama3.2:1b")

In [3]:
class redis_sac(TypedDict):
    message: Annotated[list,add]

In [4]:
def chat(state: redis_sac) -> redis_sac:
    messages = state["message"]
    response = llm.invoke(messages)
    return {"message": [AIMessage(content=response.content)]}

In [5]:
graph=StateGraph(redis_sac)


In [6]:
graph.add_node("chat",chat)
graph.add_edge(START,"chat")
graph.add_edge("chat",END)

In [11]:
REDIS_URI = "redis://:6379"

with RedisSaver.from_conn_string(REDIS_URI) as checkpointer:

    # Run once to create the required Redis indices
    checkpointer.setup()

    app = graph.compile(checkpointer=checkpointer)

    config = {
        "configurable": {
            "thread_id": "user-1"
        }
    }

    result = app.invoke(
        {
            "chat": [
                HumanMessage(content="Hello!")
            ]
        },
        config=config,
    )

    print(result["chat"][-1].content)

ConnectionError: Error 10061 connecting to localhost:6379. No connection could be made because the target machine actively refused it.

In [12]:
!redis ping

'redis' is not recognized as an internal or external command,
operable program or batch file.
